# 20 UDFs (User-Defined Functions)

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Write Python UDFs, Arrow-optimized UDFs, pandas UDFs and SQL UDFs, register them for SQL, and understand <b>why</b> built-in functions are almost always faster, so you only reach for a UDF when there is no alternative.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Why should you avoid UDFs?" is one of the most common PySpark interview questions. A Python UDF moves every row from the JVM to a Python process and back, and the optimizer can't see inside it. Yet UDFs are sometimes the right tool: calling a Python library, complex parsing, or ML scoring. Knowing the options (and their costs) is what separates a beginner from an experienced Spark developer. This lesson is <b>interview-critical</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The options, from fastest to slowest</b>
<ol><li><b>Built-in functions</b> (<code>pyspark.sql.functions</code>, SQL functions): run in the JVM, optimized by Catalyst, code-generated. Always try these first</li><li><b>SQL UDFs</b> (<code>CREATE FUNCTION ... RETURN &lt;SQL expression&gt;</code>): defined in SQL, inlined into the plan like built-ins</li><li><b>pandas UDFs</b> (<code>@pandas_udf</code>): vectorized. Data moves in Apache Arrow batches and your function works on whole pandas Series</li><li><b>Arrow-optimized Python UDFs</b> (<code>@udf(useArrow=True)</code>): row-at-a-time Python, but transferred with Arrow</li><li><b>Classic Python UDFs</b> (<code>@udf</code>): row-at-a-time, pickled serialization. Slowest</li></ol>
</div>

```
Built-in:     JVM executor ── expression evaluated in generated Java code ──> result

Python UDF:   JVM executor ──serialize rows──> Python worker ──run your function──> serialize back ──> JVM
                              (per row or per Arrow batch)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A logistics company needs to validate 30 million addresses with a Python library that has no Spark equivalent. A plain Python UDF processes them in 40 minutes. Rewritten as a pandas UDF that calls the library on whole batches, it takes 6 minutes. Meanwhile the "clean the postcode" step, which was also a UDF, is replaced by <code>upper(regexp_replace(...))</code> and drops from 8 minutes to a few seconds.
</div>

## Syntax

```python
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

@udf(returnType="string")                 # classic Python UDF
def clean(s):
    return s.strip().upper() if s is not None else None

@udf(returnType="string", useArrow=True)  # Arrow-optimized Python UDF (Spark 3.5+)
def clean_arrow(s): ...

@pandas_udf("double")                     # vectorized pandas UDF
def to_usd(amount: pd.Series, rate: pd.Series) -> pd.Series:
    return amount * rate

spark.udf.register("clean_sql", clean)    # make it callable from SQL
spark.sql("CREATE OR REPLACE TEMPORARY FUNCTION tax(x DOUBLE) RETURNS DOUBLE RETURN x * 0.2")
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates product reviews with messy text, a price and a currency.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 5-row table. One review is null.
</div>

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

reviews = spark.createDataFrame(
    [
        (1, "  Great laptop, FAST delivery!!  ", 1200.0, "USD"),
        (2, "battery died after 2 days :(", 80.0, "GBP"),
        (3, None, 15.5, "EUR"),
        (4, "Good value. Would buy again.", 300.0, "INR"),
        (5, "TERRIBLE!!! never again", 45.0, "USD"),
    ],
    "review_id INT, text STRING, price DOUBLE, currency STRING",
)
display(reviews)

## 1. A classic Python UDF

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines a UDF that counts words and classifies sentiment from keywords, then applies it.<br><br>
<b>Why it matters</b><br>This shows the basic mechanics: the decorator, the <b>return type</b>, and handling <code>None</code>. Spark passes Python <code>None</code> for null values, so the function must handle it or it will fail at runtime.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>word_count</code> and a <code>sentiment</code> per review. The null review gives <code>0</code> words and <code>unknown</code> sentiment.
</div>

In [0]:
@udf(returnType="int")
def word_count(text):
    if text is None:
        return 0
    return len(text.split())

POSITIVE = {"great", "good", "fast", "love"}
NEGATIVE = {"terrible", "died", "bad", "never"}

@udf(returnType="string")
def sentiment(text):
    if text is None:
        return "unknown"
    words = set(text.lower().replace("!", " ").replace(".", " ").replace(",", " ").split())
    score = len(words & POSITIVE) - len(words & NEGATIVE)
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"

reviews.select("review_id", "text", word_count("text").alias("word_count"), sentiment("text").alias("sentiment")) \
       .show(truncate=False)

## 2. The same logic with built-in functions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Rewrites both UDFs with native functions: <code>split</code>, <code>size</code>, <code>regexp_replace</code>, <code>array_intersect</code>.<br><br>
<b>Why it matters</b><br>Most "I need a UDF" cases can be expressed with built-ins. The result is identical, but it runs in the JVM and the optimizer understands it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same <code>word_count</code> and <code>sentiment</code> values as section 1.
</div>

In [0]:
words = F.split(F.trim(F.lower(F.regexp_replace("text", r"[!.,]", " "))), r"\s+")
pos = F.array(*[F.lit(w) for w in POSITIVE])
neg = F.array(*[F.lit(w) for w in NEGATIVE])
score = F.size(F.array_intersect(F.array_distinct(words), pos)) - F.size(F.array_intersect(F.array_distinct(words), neg))

reviews.select(
    "review_id",
    F.when(F.col("text").isNull(), 0).otherwise(F.size(F.split(F.trim("text"), r"\s+"))).alias("word_count"),
    F.when(F.col("text").isNull(), "unknown")
     .when(score > 0, "positive").when(score < 0, "negative").otherwise("neutral").alias("sentiment"),
).show()

## 3. Edge case: the wrong return type gives silent nulls

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Declares a UDF as returning <code>int</code> but returns a Python <code>float</code>, then repeats it with the default return type.<br><br>
<b>Why it matters</b><br>For classic Python UDFs, if the returned value doesn't match the declared type, Spark may return <b>null instead of raising an error</b>. The default return type is <code>string</code>. This is a nasty silent bug. (Arrow-optimized UDFs are stricter and try to convert or raise an error.)
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>wrong_type</code> is null (or an error, depending on the runtime), <code>default_type</code> is the string <code>"2400.0"</code>, and <code>correct</code> is <code>2400.0</code> as a double.
</div>

In [0]:
@udf(returnType="int")
def double_price_wrong(p):
    return p * 2          # returns a float, but the UDF promises an int

@udf                       # no return type: defaults to StringType
def double_price_default(p):
    return p * 2

@udf(returnType="double")
def double_price(p):
    return p * 2 if p is not None else None

try:
    out = reviews.select(
        "price",
        double_price_wrong("price").alias("wrong_type"),
        double_price_default("price").alias("default_type"),
        double_price("price").alias("correct"),
    )
    out.show()
    out.printSchema()
except Exception as e:
    print("Failed:", type(e).__name__, "-", str(e).splitlines()[0][:160])

## 4. pandas UDFs: vectorized and much faster

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Converts prices to USD with a pandas UDF that receives whole pandas Series (batches of rows) instead of one value at a time.<br><br>
<b>Why it matters</b><br>Data moves in Arrow batches (thousands of rows each), and your code uses vectorized pandas/NumPy operations. That removes most of the per-row Python overhead. Use type hints (<code>pd.Series -&gt; pd.Series</code>) to define the UDF type.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>price_usd</code> column, for example 101.6 for 80 GBP, and 3.6 for 300 INR.
</div>

In [0]:
RATES = {"USD": 1.0, "GBP": 1.27, "EUR": 1.08, "INR": 0.012}

@pandas_udf("double")
def to_usd(price: pd.Series, currency: pd.Series) -> pd.Series:
    # Both inputs are pandas Series of the same length (one Arrow batch).
    return (price * currency.map(RATES)).round(2)

reviews.select("price", "currency", to_usd("price", "currency").alias("price_usd")).show()

## 5. Registering UDFs for SQL, and SQL UDFs

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Registers a Python UDF so SQL can call it, then defines a pure SQL UDF with <code>CREATE TEMPORARY FUNCTION</code>.<br><br>
<b>Why it matters</b><br>Analysts in Databricks SQL can use your logic. A <b>SQL UDF</b> whose body is a SQL expression is inlined into the query plan, so it runs at built-in speed. On Databricks you can also create permanent SQL functions in Unity Catalog (<code>CREATE FUNCTION catalog.schema.name ...</code>), which are governed and shareable. That is covered in <code>06_unity_catalog</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
SQL results with <code>sentiment_sql</code> from the registered Python UDF, and <code>price_with_tax</code> from the SQL UDF (for example 1440.0 for 1200.0).
</div>

In [0]:
spark.udf.register("sentiment_sql", sentiment)
spark.sql("CREATE OR REPLACE TEMPORARY FUNCTION with_tax(price DOUBLE) RETURNS DOUBLE RETURN round(price * 1.2, 2)")

reviews.createOrReplaceTempView("reviews_v")
spark.sql("""
    SELECT review_id, sentiment_sql(text) AS sentiment, price, with_tax(price) AS price_with_tax
    FROM reviews_v
""").show()

## 6. Measure it: built-in vs Python UDF vs pandas UDF

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Applies the same calculation to 2 million rows four ways and times each. <code>noop</code> output forces full execution without writing anything.<br><br>
<b>Why it matters</b><br>Numbers make the argument for you in interviews and design reviews. The fixed cost of starting Python workers and moving data is paid by every UDF type, which is why built-ins win so clearly.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Four timings. The built-in version is the fastest by a wide margin (often 5 to 10 times faster). Among the UDFs, the pandas and Arrow versions are usually faster than the classic one, and the gap grows with more rows and heavier per-row work. Exact times vary between runs and compute types, so write yours down.
</div>

In [0]:
import time

big = spark.range(2_000_000).withColumn("x", F.col("id") * 1.0)

@udf("double")
def plus_tax_udf(x):
    return x * 1.2 + 1

@udf("double", useArrow=True)
def plus_tax_arrow(x):
    return x * 1.2 + 1

@pandas_udf("double")
def plus_tax_pandas(x: pd.Series) -> pd.Series:
    return x * 1.2 + 1

def timed(label, col):
    start = time.time()
    big.select(col.alias("y")).write.format("noop").mode("overwrite").save()
    print(f"{label:<22} {time.time() - start:6.2f} s")

timed("built-in", F.col("x") * 1.2 + 1)
timed("pandas UDF", plus_tax_pandas("x"))
timed("Arrow Python UDF", plus_tax_arrow("x"))
timed("classic Python UDF", plus_tax_udf("x"))

## Under the hood

```
built-in:   *(1) Project [((x * 1.2) + 1.0) AS y]           <- inside whole-stage codegen
Python UDF:      BatchEvalPython [plus_tax_udf(x)]          <- rows pickled to a Python worker
Arrow UDF:       ArrowEvalPython [plus_tax_arrow(x)]        <- Arrow batches, row-at-a-time function
pandas UDF:      ArrowEvalPython [plus_tax_pandas(x)]       <- Arrow batches, vectorized function
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> UDFs are evaluated row by row inside each partition, with no shuffle of their own.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> but the optimizer treats them as a **black box**.

Why UDFs are slower:
- **Serialization:** each value is converted between JVM and Python formats, twice
- **Process boundary:** a separate Python worker runs beside each executor task
- **No optimization:** Catalyst can't push filters through a UDF, can't constant-fold it, and whole-stage codegen stops at the UDF
- **No statistics:** the optimizer can't estimate the selectivity of a filter on a UDF

Compare the plans below: the built-in stays in one `*(1)` codegen stage, while the UDFs add `BatchEvalPython` / `ArrowEvalPython` nodes.

In [0]:
print("---- built-in ----")
big.select((F.col("x") * 1.2 + 1).alias("y")).explain()
print("---- classic Python UDF ----")
big.select(plus_tax_udf("x").alias("y")).explain()
print("---- pandas UDF ----")
big.select(plus_tax_pandas("x").alias("y")).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: UDF column is all null</b><br>
The return type doesn't match what the function returns (for example int vs float), or you forgot the return type and it defaulted to string. Declare it and return that exact type.<br><br>
**⛔ Problem: <code>TypeError: 'NoneType' ...</code> inside the UDF**<br>Spark passes <code>None</code> for nulls. Handle it at the top of the function.<br><br>
**⛔ Problem: the job is slow and the plan shows <code>BatchEvalPython</code>**<br>A row-at-a-time Python UDF. Replace it with built-ins, a SQL UDF, or a pandas UDF.<br><br>
<b>⛔ Problem: a filter after a UDF doesn't skip any data</b><br>Filters on UDF results can't be pushed down. Filter on raw columns first, then apply the UDF to fewer rows.<br><br>
**⛔ Problem: <code>ModuleNotFoundError</code> inside a UDF**<br>The library is missing on the workers. On serverless notebooks, install with <code>%pip install &lt;lib&gt;</code> at the top of the notebook (it installs for the notebook's environment).<br><br>
<b>⛔ Problem: random or time-based UDF gives inconsistent results</b><br>Spark may re-run UDFs. Mark them with <code>.asNondeterministic()</code>, or better, use built-in <code>rand()</code> and <code>current_timestamp()</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Why should you avoid Python UDFs in PySpark?</b><br>
Every row must be serialized from the JVM to a Python worker and back, which is expensive. The optimizer also treats the UDF as a black box, so it can't push filters through it, can't include it in whole-stage code generation, and has no statistics about it. Built-in functions avoid all of this.<br><br>

<b>🎤 2. What is a pandas UDF and why is it faster than a regular UDF?</b><br>
A vectorized UDF. Spark sends data in Apache Arrow batches, and the function processes a whole pandas Series at once with vectorized operations. That removes the per-row serialization and Python call overhead.<br><br>

<b>🎤 3. When is a UDF still the right choice?</b><br>
When the logic can't be expressed with built-ins or SQL, for example calling a Python library for parsing, validation, encryption, or ML model inference. Then I prefer a pandas UDF and I filter the data first so fewer rows reach it.<br><br>

<b>🎤 4. What happens if a UDF returns a different type than declared?</b><br>
For classic Python UDFs, the values may silently become null. The default return type is string. Arrow-optimized UDFs try to convert the value and raise an error if they can't.<br><br>

<b>🎤 5. How do you use a Python UDF in SQL?</b><br>
Register it with <code>spark.udf.register("name", func)</code>. For logic that can be written in SQL, a SQL UDF (<code>CREATE FUNCTION ... RETURN expr</code>) is better, because it is inlined and runs at built-in speed. In Unity Catalog it can also be permanent and governed.<br><br>

<b>🎤 6. What is an Arrow-optimized Python UDF?</b><br>
A regular row-at-a-time Python UDF (<code>useArrow=True</code>) that uses Arrow instead of pickle to move data between the JVM and Python. It is faster than a classic UDF and has stricter type handling, but slower than a pandas UDF.<br><br>

<b>🎤 7. How do you spot a UDF in a query plan?</b><br>
Look for <code>BatchEvalPython</code> (classic Python UDF) or <code>ArrowEvalPython</code> (Arrow or pandas UDF) nodes. They break the whole-stage codegen stage.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A team uses a Python UDF to convert a string to upper case and remove spaces on a 1 billion row table. What is the best improvement?</b><br>
A. Convert the UDF to a pandas UDF<br>
B. Replace it with built-in functions <code>upper</code> and <code>regexp_replace</code><br>
C. Increase the number of shuffle partitions<br>
D. Cache the table before applying the UDF<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Built-ins avoid Python serialization entirely and are optimized. A pandas UDF helps, but is still slower than built-ins.</details><br><br>

<b>Q2. Which statement creates a SQL function that can be used in Databricks SQL queries and is inlined by the optimizer?</b><br>
A. <code>spark.udf.register("f", lambda x: x * 2)</code><br>
B. <code>CREATE FUNCTION f(x DOUBLE) RETURNS DOUBLE RETURN x * 2</code><br>
C. <code>@pandas_udf("double")</code><br>
D. <code>CREATE PROCEDURE f(x DOUBLE)</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> A SQL UDF with a SQL expression body. A registers a Python UDF, and C is a vectorized Python UDF.</details><br><br>

<b>Q3. Why can a filter on the result of a Python UDF be slower than a filter on the raw column?</b><br>
A. UDF results are always cached<br>
B. The optimizer can't push the filter down to the data source<br>
C. UDFs force a shuffle<br>
D. UDFs run only on the driver<br>
<details><summary><b>Show answer</b></summary><b>B.</b> The optimizer can't reason about the UDF, so it can't push the filter into the scan to skip data.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>reviews</code>:
<ol><li>Write a Python UDF <code>mask_text</code> that replaces every digit with <code>#</code> (handle nulls), and apply it</li><li>Rewrite <code>mask_text</code> with a built-in function and confirm the results match</li><li>Write a pandas UDF <code>price_band</code> returning <code>"low"</code> (< 50), <code>"mid"</code> (50 to 500) or <code>"high"</code> (> 500)</li><li>Create a SQL UDF <code>discount(price DOUBLE, pct DOUBLE)</code> and use it in a <code>spark.sql</code> query</li><li>Look at the plans of tasks 1 and 2 and write down the node that shows the UDF</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
import re

# 1. Python UDF with null handling
@udf("string")
def mask_text(s):
    return None if s is None else re.sub(r"\d", "#", s)

masked = reviews.select("review_id", mask_text("text").alias("udf_version"),
                        F.regexp_replace("text", r"\d", "#").alias("builtin_version"))   # 2. built-in
masked.show(truncate=False)
print("Results match:", masked.filter(~F.col("udf_version").eqNullSafe(F.col("builtin_version"))).count() == 0)

# 3. Vectorized banding with pandas
@pandas_udf("string")
def price_band(p: pd.Series) -> pd.Series:
    return pd.cut(p, bins=[-float("inf"), 50, 500, float("inf")], labels=["low", "mid", "high"], right=False).astype(str)

reviews.select("price", price_band("price").alias("band")).show()

# 4. SQL UDF
spark.sql("CREATE OR REPLACE TEMPORARY FUNCTION discount(price DOUBLE, pct DOUBLE) RETURNS DOUBLE RETURN round(price * (1 - pct / 100), 2)")
spark.sql("SELECT review_id, price, discount(price, 10) AS discounted FROM reviews_v").show()

# 5. Task 1 shows BatchEvalPython in the plan, task 2 shows only a Project with regexp_replace.
reviews.select(mask_text("text")).explain()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Order of preference: built-ins > SQL UDF > pandas UDF > Arrow Python UDF > classic Python UDF</li><li>Python UDFs serialize every row between the JVM and Python, and are a black box to Catalyst</li><li>Always declare the return type (default is string) and handle <code>None</code></li><li>pandas UDFs work on Arrow batches as pandas Series, which is much faster</li><li><code>spark.udf.register</code> exposes Python UDFs to SQL. SQL UDFs are inlined and fast</li><li>In plans, look for <code>BatchEvalPython</code> / <code>ArrowEvalPython</code></li><li>Filter before applying a UDF so fewer rows cross into Python</li></ul>
</div>

| Task | Code |
|---|---|
| Python UDF | `@udf("string")` |
| Arrow UDF | `@udf("string", useArrow=True)` |
| pandas UDF | `@pandas_udf("double")` with `pd.Series` type hints |
| Register for SQL | `spark.udf.register("name", f)` |
| SQL UDF | `CREATE OR REPLACE TEMPORARY FUNCTION f(x DOUBLE) RETURNS DOUBLE RETURN x * 2` |
| Benchmark | `df.write.format("noop").mode("overwrite").save()` |

## Git commit

```
git add 01_spark_basics/20_udfs.ipynb
git commit -m "add 20 udfs notebook"
```